In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.subplots import make_subplots

In [2]:
# Carregamento do dataset
df = pd.read_excel('../../Data/Raw/populacao.xls', skiprows=1)

In [3]:
df.columns

Index(['BRASIL E UNIDADES DA FEDERAÇÃO', 'POPULAÇÃO ESTIMADA', 'Unnamed: 2'], dtype='object')

In [ ]:
df = df.rename(columns={'BRASIL E UNIDADES DA FEDERAÇÃO': 'Regiao', 'POPULAÇÃO ESTIMADA': 'Populacao'})

In [5]:
df.head

<bound method NDFrame.head of                                                Regiao    Populacao Unnamed: 2
0                                              Brasil  213421037.0        NaN
1                                               Norte   18801282.0        NaN
2                                            Rondônia    1751950.0        NaN
3                                                Acre     884372.0        NaN
4                                            Amazonas    4321616.0        NaN
5                                             Roraima     738772.0        NaN
6                                                Pará    8711196.0        NaN
7                                               Amapá     806517.0        NaN
8                                           Tocantins    1586859.0        NaN
9                                            Nordeste   57244485.0        NaN
10                                           Maranhão    7018211.0        NaN
11                                

In [ ]:
df_top10= df.sort_values(by='Regiao', ascending=False)
df_top10.head(10)

,Regiao,Populacao,Unnamed: 2
8,Tocantins,1586859.0,NaN
23,São Paulo,46081801.0,NaN
24,Sul,31310809.0,NaN
19,Sudeste,88825643.0,NaN
17,Sergipe,2299425.0,NaN
26,Santa Catarina,8187029.0,(1)
5,Roraima,738772.0,NaN
2,Rondônia,1751950.0,NaN
22,Rio de Janeiro,17223547.0,NaN
27,Rio Grande do Sul,11233263.0,NaN


In [ ]:
fig = px.bar(
    df_top10,
    x='Regiao',
    y='Populacao',
    color='Regiao',
    text='Populacao',
)


In [12]:
total_brasil = df.loc[df['Regiao'] == 'Brasil', 'Populacao'].values[0]

In [27]:
import plotly.express as px

fig = px.bar(df_top10, x='Regiao', y='Populacao', color='Regiao')

fig.update_layout(
    title={
        'text': 'Top 10 Estados Mais Populosos',
        'y': 0.95,
        'x': 0.43,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24, color='#2C3E50', family='Arial Black')
    },
    annotations=[
        dict(
            text='<b style="font-size:18px; color:#2C3E50">BRASIL</b>',
            x=0.5, y=0.58,
            xref='paper', yref='paper',
            font=dict(family='Arial Black'),
            showarrow=False
        ),
        dict(
            text=f'<b style="font-size:32px; color:#E74C3C">{total_brasil/1_000_000:.1f}M</b>',
            x=0.5, y=0.50,
            xref='paper', yref='paper',
            font=dict(family='Arial Black'),
            showarrow=False
        ),
        dict(
            text='<span style="font-size:13px; color:#34495E">habitantes</span>',
            x=0.5, y=0.42,
            xref='paper', yref='paper',
            font=dict(family='Arial'),
            showarrow=False
        ),
    ],
    height=750,
    width=1200,

    # 🔹 Legenda aprimorada
    legend=dict(
        title='',
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.08,
        font=dict(size=14, family='Arial', color='#2C3E50'),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#BDC3C7',
        borderwidth=1,
        itemsizing='trace',
        traceorder='normal',
        tracegroupgap=3
    ),

    paper_bgcolor='#ECF0F1',
    plot_bgcolor='#ECF0F1',
    margin=dict(l=80, r=120, t=100, b=60),
    font=dict(size=13, family='Arial', color='#2C3E50')
)

fig.show()

In [15]:
COLORS = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8',
    '#F7DC6F', '#BB8FCE', '#85C1E2', '#F8B739', '#52B788'
]

# ⚙️ Configurações de Gráfico
CHART_CONFIG = {
    'height': 850,
    'width': 1400,
    'horizontal_spacing': 0.12,
    'vertical_spacing': 0.15
}

STYLE_CONFIG = {
    'paper_bgcolor': '#ECF0F1',
    'plot_bgcolor': '#FFFFFF',
    'grid_color': '#E5E8E8',
    'primary_color': '#2C3E50',
    'accent_color': '#E74C3C'
}

# 🔧 Ajusta e prepara dados
def preparar_dados(df, n_top=10):
    # Garantir que as colunas tenham nomes uniformes
    df = df.rename(columns={
        'BRASIL E UNIDADES DA FEDERAÇÃO': 'Regiao',
        'BRASIL E UNIDADES DA FEDERACAO': 'Regiao',  # alternativa
        'População': 'Populacao'
    })
    
    # 🔹 Pega o total da linha "Brasil"
    total_brasil = df.loc[df['Regiao'].str.lower() == 'brasil', 'Populacao'].values[0]
    
    # 🔹 Remove Brasil e pega os 10 maiores estados
    df_estados = df.loc[df['Regiao'].str.lower() != 'brasil'].copy()
    top_n = df_estados.nlargest(n_top, 'Populacao').copy()
    
    # Calcula percentuais e textos auxiliares
    top_n['Percentual'] = (top_n['Populacao'] / total_brasil * 100).round(1)
    top_n['Label_Legenda'] = top_n['Regiao'] + ' (' + top_n['Percentual'].astype(str) + '%)'
    top_n['Pop_Milhoes'] = (top_n['Populacao'] / 1_000_000).round(1).astype(str) + 'M'
    
    return top_n, total_brasil


# 🥧 Gráfico de Pizza
def criar_grafico_pizza(top_10):
    pie = go.Pie(
        labels=top_10['Label_Legenda'],
        values=top_10['Populacao'],
        hole=0.55,
        marker=dict(colors=COLORS, line=dict(color='#FFFFFF', width=2)),
        textinfo='none',
        hoverinfo='label+value+percent',
        hovertemplate='<b>%{label}</b><br>População: %{value:,.0f}<extra></extra>',
        pull=[0.03] * len(top_10),
        name='Distribuição',
        showlegend=False
    )
    return pie


# 📊 Barras Verticais
def criar_grafico_barras_verticais(top_10):
    return go.Bar(
        x=top_10['Regiao'],
        y=top_10['Populacao'],
        name='População',
        marker_color=COLORS,
        text=top_10['Pop_Milhoes'],
        textposition='outside',
        textfont=dict(size=11, family='Arial'),
        hovertemplate='<b>%{x}</b><br>População: %{y:,.0f}<extra></extra>'
    )


# 📈 Linha
def criar_grafico_linha(top_10):
    return go.Scatter(
        x=top_10['Regiao'],
        y=top_10['Populacao'],
        mode='lines+markers',
        name='Tendência',
        line=dict(color=STYLE_CONFIG['accent_color'], width=3, shape='spline'),
        marker=dict(size=10, color='#C0392B', line=dict(width=2, color='white')),
        hovertemplate='<b>%{x}</b><br>População: %{y:,.0f}<extra></extra>',
        fill='tozeroy',
        fillcolor='rgba(231, 76, 60, 0.1)'
    )


# 📊 Barras Horizontais
def criar_grafico_barras_horizontais(top_10):
    top_10_inv = top_10.iloc[::-1]
    return go.Bar(
        y=top_10_inv['Regiao'],
        x=top_10_inv['Populacao'],
        orientation='h',
        marker_color=COLORS[::-1],
        text=top_10_inv['Pop_Milhoes'],
        textposition='outside',
        textfont=dict(size=11, family='Arial'),
        hovertemplate='<b>%{y}</b><br>População: %{x:,.0f}<extra></extra>'
    )


# 🧩 Monta o Dashboard Final
def criar_dashboard(df):
    top_10, total = preparar_dados(df)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Distribuição Percentual', 'Comparativo por Estado', 'Curva Populacional', 'Ranking (Maior → Menor)'),
        specs=[[{"type": "pie"}, {"type": "bar"}],
                [{"type": "scatter"}, {"type": "bar"}]],
        horizontal_spacing=CHART_CONFIG['horizontal_spacing'],
        vertical_spacing=CHART_CONFIG['vertical_spacing']
    )
    
    fig.add_trace(criar_grafico_pizza(top_10), row=1, col=1)
    fig.add_trace(criar_grafico_barras_verticais(top_10), row=1, col=2)
    fig.add_trace(criar_grafico_linha(top_10), row=2, col=1)
    fig.add_trace(criar_grafico_barras_horizontais(top_10), row=2, col=2)
    
    # 🔸 Anotação com total no centro superior
    fig.add_annotation(
        text=f'<b>{total/1_000_000:.1f}M</b><br><span style="font-size:12px">habitantes</span>',
        x=0.19, y=0.83,
        xref='paper', yref='paper',
        font=dict(size=18, color=STYLE_CONFIG['primary_color'], family='Arial Black'),
        showarrow=False,
        align='center'
    )
    
    # 🔧 Layout
    fig.update_layout(
        height=CHART_CONFIG['height'],
        width=CHART_CONFIG['width'],
        title={
            'text': '<b>Dashboard Populacional do Brasil - Top 10 Estados</b>',
            'y': 0.98,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=26, color=STYLE_CONFIG['primary_color'], family='Arial Black')
        },
        showlegend=False,
        paper_bgcolor=STYLE_CONFIG['paper_bgcolor'],
        plot_bgcolor=STYLE_CONFIG['plot_bgcolor'],
        font=dict(size=12, family='Arial', color=STYLE_CONFIG['primary_color']),
        margin=dict(l=80, r=100, t=120, b=80),
        hovermode='closest'
    )
    
    fig.update_xaxes(showgrid=True, gridcolor=STYLE_CONFIG['grid_color'], tickangle=-45)
    fig.update_yaxes(showgrid=True, gridcolor=STYLE_CONFIG['grid_color'])
    
    fig.update_traces(
        hoverlabel=dict(
            bgcolor="white",
            font_size=13,
            font_family="Arial",
            bordercolor=STYLE_CONFIG['primary_color']
        )
    )
    return fig


# 🔹 Execução
fig = criar_dashboard(df)
fig.show()